In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/centralVN_dataWeather.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/aqi_centralVN_daily.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/aqi_southVN_daily.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/northVN_dataThoiTiet.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/southVN_dataAIR.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/centralVN.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/southVN_dataWeather.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/centralVN_dataAIR.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/northVN_dataAIR.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/southVN.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/northVN.csv
/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/aqi_northVN_daily.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import tensorflow as tf
from keras.models import Sequential
from keras.layers import LSTM, Dropout, Dense, Input

import matplotlib.pyplot as plt

2026-05-30 09:37:00.344285: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780133820.538219      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780133820.596801      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780133821.059855      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780133821.059894      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780133821.059897      57 computation_placer.cc:177] computation placer alr

# 1.Load data

In [2]:
df = pd.read_csv('/kaggle/input/datasets/fasgadhsxnzmjj/mmd-dataset/northVN.csv')

In [3]:
df = df[df['district'] == 'Thanh Xuan']

In [4]:
df.head()

,district,city,time,temperature_2m (°C),relative_humidity_2m (%),apparent_temperature (°C),precipitation (mm),rain (mm),cloud_cover (%),cloud_cover_low (%),...,pm2_5 (μg/m³),carbon_monoxide (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),aerosol_optical_depth (),dust (μg/m³),uv_index (),uv_index_clear_sky (),aqi_h
214655,Thanh Xuan,Ha Noi,2022-08-04 07:00:00,28.3,86,34.8,0.0,0.0,40,0,...,41.7,713.0,38.6,20.1,22.0,0.34,0.0,0.75,0.85,83
214656,Thanh Xuan,Ha Noi,2022-08-04 08:00:00,29.5,82,36.5,0.0,0.0,99,0,...,29.9,658.0,32.8,21.0,50.0,0.39,0.0,2.20,2.50,74
214657,Thanh Xuan,Ha Noi,2022-08-04 09:00:00,30.6,79,38.1,0.0,0.0,100,4,...,30.4,581.0,24.4,22.5,90.0,0.47,0.0,3.90,4.85,71
214658,Thanh Xuan,Ha Noi,2022-08-04 10:00:00,31.8,75,39.8,0.0,0.0,99,10,...,35.5,503.0,15.8,24.0,137.0,0.57,0.0,5.20,7.15,71
214659,Thanh Xuan,Ha Noi,2022-08-04 11:00:00,32.5,72,41.2,0.0,0.0,92,4,...,44.0,490.0,13.5,24.4,160.0,0.68,0.0,5.45,8.55,72


# 2.Data cleaning

In [5]:
print(df.columns)

Index(['district', 'city', 'time', 'temperature_2m (°C)',
       'relative_humidity_2m (%)', 'apparent_temperature (°C)',
       'precipitation (mm)', 'rain (mm)', 'cloud_cover (%)',
       'cloud_cover_low (%)', 'cloud_cover_mid (%)', 'cloud_cover_high (%)',
       'wind_speed_10m (km/h)', 'wind_speed_100m (km/h)',
       'wind_direction_10m (°)', 'soil_temperature_0_to_7cm (°C)',
       'soil_temperature_7_to_28cm (°C)', 'soil_temperature_28_to_100cm (°C)',
       'soil_temperature_100_to_255cm (°C)', 'soil_moisture_0_to_7cm (m³/m³)',
       'soil_moisture_7_to_28cm (m³/m³)', 'soil_moisture_28_to_100cm (m³/m³)',
       'soil_moisture_100_to_255cm (m³/m³)', 'pm10 (μg/m³)', 'pm2_5 (μg/m³)',
       'carbon_monoxide (μg/m³)', 'nitrogen_dioxide (μg/m³)',
       'sulphur_dioxide (μg/m³)', 'ozone (μg/m³)', 'aerosol_optical_depth ()',
       'dust (μg/m³)', 'uv_index ()', 'uv_index_clear_sky ()', 'aqi_h'],
      dtype='object')


In [7]:
# chọn ra những cột quan trọng
selected_columns = [
    'district', 'time',

    # pollutants
    'pm2_5 (μg/m³)', 'pm10 (μg/m³)',
    'carbon_monoxide (μg/m³)',
    'nitrogen_dioxide (μg/m³)',
    'sulphur_dioxide (μg/m³)',
    'ozone (μg/m³)',
    'aerosol_optical_depth ()',

    'aqi_h'
]

df = df[selected_columns]

print(df.shape)

(30665, 10)


In [8]:
df['time'] = pd.to_datetime(df['time'])

In [9]:
df.isnull().sum()

district                    0
time                        0
pm2_5 (μg/m³)               0
pm10 (μg/m³)                0
carbon_monoxide (μg/m³)     0
nitrogen_dioxide (μg/m³)    0
sulphur_dioxide (μg/m³)     0
ozone (μg/m³)               0
aerosol_optical_depth ()    0
aqi_h                       0
dtype: int64

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 30665 entries, 214655 to 245319
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   district                  30665 non-null  object        
 1   time                      30665 non-null  datetime64[ns]
 2   pm2_5 (μg/m³)             30665 non-null  float64       
 3   pm10 (μg/m³)              30665 non-null  float64       
 4   carbon_monoxide (μg/m³)   30665 non-null  float64       
 5   nitrogen_dioxide (μg/m³)  30665 non-null  float64       
 6   sulphur_dioxide (μg/m³)   30665 non-null  float64       
 7   ozone (μg/m³)             30665 non-null  float64       
 8   aerosol_optical_depth ()  30665 non-null  float64       
 9   aqi_h                     30665 non-null  int64         
dtypes: datetime64[ns](1), float64(7), int64(1), object(1)
memory usage: 2.6+ MB


In [11]:
df = df.sort_values(['district', 'time']).reset_index(drop=True)
df.head()

,district,time,pm2_5 (μg/m³),pm10 (μg/m³),carbon_monoxide (μg/m³),nitrogen_dioxide (μg/m³),sulphur_dioxide (μg/m³),ozone (μg/m³),aerosol_optical_depth (),aqi_h
0,Thanh Xuan,2022-08-04 07:00:00,41.7,60.1,713.0,38.6,20.1,22.0,0.34,83
1,Thanh Xuan,2022-08-04 08:00:00,29.9,43.3,658.0,32.8,21.0,50.0,0.39,74
2,Thanh Xuan,2022-08-04 09:00:00,30.4,44.0,581.0,24.4,22.5,90.0,0.47,71
3,Thanh Xuan,2022-08-04 10:00:00,35.5,51.2,503.0,15.8,24.0,137.0,0.57,71
4,Thanh Xuan,2022-08-04 11:00:00,44.0,63.3,490.0,13.5,24.4,160.0,0.68,72


# 3. Model

## Chọn feature

In [12]:
features = [
    'pm2_5 (μg/m³)', 'pm10 (μg/m³)',
    'carbon_monoxide (μg/m³)',
    'nitrogen_dioxide (μg/m³)',
    'sulphur_dioxide (μg/m³)',
    'ozone (μg/m³)',
    'aerosol_optical_depth ()',

    'aqi_h'
]

target = 'aqi_h'

## Train test split

In [14]:
split = int(len(df) * 0.8)

train = df.iloc[:split].copy()
test = df.iloc[split:].copy()

## Scale data

In [15]:
from sklearn.preprocessing import StandardScaler

scaler_x = StandardScaler()
scaler_y = StandardScaler()

train[features] = scaler_x.fit_transform(
    train[features]
)

test[features] = scaler_x.transform(
    test[features]
)

train[[target]] = scaler_y.fit_transform(
    train[[target]]
)

test[[target]] = scaler_y.transform(
    test[[target]]
)

## Tạo SEQUENCE (MULTI-STEP)

In [16]:
def create_sequences_multi(
    data,
    features,
    target,
    window=24,
    horizon=24
):

    x = []
    y = []

    values = data[features + [target]].values

    for i in range(
        len(values) - window - horizon
    ):

        x.append(
            values[i:i+window, :-1]
        )

        y.append(
            values[
                i+window:
                i+window+horizon,
                -1
            ]
        )

    return np.array(x), np.array(y)

x_train, y_train = create_sequences_multi(
    train,
    features,
    target
)

x_test, y_test = create_sequences_multi(
    test,
    features,
    target
)

print("x_train:", x_train.shape)
print("y_train:", y_train.shape)

print("x_test:", x_test.shape)
print("y_test:", y_test.shape)

x_train: (24484, 24, 8)
y_train: (24484, 24)
x_test: (6085, 24, 8)
y_test: (6085, 24)


## Buid model LSTM

In [17]:
from keras.models import Sequential
from keras.layers import (
    LSTM,
    Dense,
    Dropout,
    Input
)

model = Sequential([

    Input(
        shape=(24, len(features))
    ),

    LSTM(
        64,
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(32),

    Dropout(0.2),

    Dense(
        32,
        activation='relu'
    ),

    Dense(24)

])

2026-05-30 11:59:14.090861: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1780142354.296838      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780142354.358465      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780142354.869993      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780142354.870021      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780142354.870024      58 computation_placer.cc:177] computation placer alr

## Compile Model

In [18]:
model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

## EarlyStopping

In [20]:
from keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

## Train

In [21]:
history = model.fit(
    x_train,
    y_train,

    validation_data=(
        x_test,
        y_test
    ),

    epochs=50,

    batch_size=128,

    callbacks=[
        early_stop
    ]
)

Epoch 1/50


I0000 00:00:1780142385.215125     155 cuda_dnn.cc:529] Loaded cuDNN version 91002


192/192 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.4962 - mae: 0.5145 - val_loss: 0.4329 - val_mae: 0.4932
Epoch 2/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.3193 - mae: 0.3919 - val_loss: 0.3961 - val_mae: 0.4549
Epoch 3/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2963 - mae: 0.3724 - val_loss: 0.3672 - val_mae: 0.4295
Epoch 4/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2819 - mae: 0.3611 - val_loss: 0.3582 - val_mae: 0.4220
Epoch 5/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2673 - mae: 0.3511 - val_loss: 0.3594 - val_mae: 0.4273
Epoch 6/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2555 - mae: 0.3433 - val_loss: 0.3822 - val_mae: 0.4326
Epoch 7/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2446 - mae: 0.3364 - val_loss: 0.3880 - val_mae: 0.4291
Epoch 8/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 0.2297 - mae: 0.3262 - val_loss: 0.4345 - val_mae: 0.4510
Epoch 9/50
192/192 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.2167

## Dự đoán

In [22]:
pred = model.predict(x_test)

191/191 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


## Inverse Scale

In [24]:
y_test_inv = scaler_y.inverse_transform(
    y_test.reshape(-1,1)
).reshape(
    y_test.shape
)

pred_inv = scaler_y.inverse_transform(
    pred.reshape(-1,1)
).reshape(
    pred.shape
)

## Evaluate

In [25]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

mae = mean_absolute_error(
    y_test_inv.flatten(),
    pred_inv.flatten()
)

mse = mean_squared_error(
    y_test_inv.flatten(),
    pred_inv.flatten()
)

rmse = np.sqrt(mse)

r2 = r2_score(
    y_test_inv.flatten(),
    pred_inv.flatten()
)

print("MAE :", round(mae,4))
print("MSE :", round(mse,4))
print("RMSE:", round(rmse,4))
print("R2  :", round(r2,4))

MAE : 0.422
MSE : 0.3582
RMSE: 0.5985
R2  : 0.7018
